=============================================================================
## УСТАНОВКА ЗАВИСИМОСТЕЙ
=============================================================================

In [1]:
!pip install mlflow torch scikit-learn numpy --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.0/814.0 kB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.9 MB/s eta 0:00:00


=============================================================================
## ИМПОРТ НЕОБХОДИМЫХ ДАННЫХ
=============================================================================

In [18]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import numpy as np
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score

=============================================================================
## АРХИТЕКТУРА НЕЙРОННОЙ СЕТИ (FCN)
=============================================================================

In [19]:
class ResidualBlock(nn.Module):
    """
    Residual-блок для табличных данных.
    Схема: x → LayerNorm → Linear → GELU → Dropout → Linear → Dropout → + x
    LayerNorm стабильнее BatchNorm на табличных данных при малых батчах.
    GELU мягче ReLU и показывает лучшие результаты на регрессии.
    """

    def __init__(self, dim: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)  # skip connection


class FCNRegressor(nn.Module):
    """
    Полносвязная нейронная сеть с residual-блоками для регрессии зарплат.

    Архитектура:
        Input → Linear (stem)
              → N × ResidualBlock
              → LayerNorm → Linear(1)

    Параметры:
        input_dim:   размерность входа
        hidden_dim:  размер скрытого пространства (одинаковый для всех блоков)
        n_blocks:    количество residual-блоков
        dropout:     вероятность дропаута внутри блоков
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 256,
        n_blocks: int = 4,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        # stem: проецируем вход в скрытое пространство
        self.stem = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )

        # стек residual-блоков
        self.blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)]
        )

        # head: финальная нормализация + линейный выход
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 1),
        )

        # инициализация весов
        self._init_weights()

    def _init_weights(self) -> None:
        """He-инициализация для линейных слоёв."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.blocks(x)
        return self.head(x).squeeze(1)

=============================================================================
## ОСНОВНОЙ КЛАСС ТРЕНЕРА
=============================================================================

In [20]:
@dataclass
class FCNTrainer:
    """
    Тренер для FCN-регрессии зарплат HH.ru с MLflow-трекингом.

    Атрибуты:
        x_path:          путь к X.npy
        y_path:          путь к y.npy
        model_name:      имя модели для MLflow (формат: fio_fcn)
        mlflow_uri:      адрес MLflow-сервера
        experiment_name: название эксперимента в MLflow
        hidden_dim:      размер скрытого слоя
        n_blocks:        количество residual-блоков
        dropout:         dropout в residual-блоках
        lr:              максимальный LR для OneCycleLR
        weight_decay:    L2-регуляризация для AdamW
        batch_size:      размер батча
        max_epochs:      максимум эпох (early stopping остановит раньше)
        patience:        эпох без улучшения до остановки
        test_size:       доля тестовой выборки
        random_state:    seed для воспроизводимости
    """

    x_path: str
    y_path: str
    model_name: str
    mlflow_uri: str = "http://kamnsv.com:55000/"
    experiment_name: str = "LIne Regression HH"
    hidden_dim: int = 256
    n_blocks: int = 4
    dropout: float = 0.1
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 512
    max_epochs: int = 500
    patience: int = 30
    test_size: float = 0.2
    random_state: int = 42

    _scaler: Optional[StandardScaler] = field(default=None, init=False, repr=False)
    _best_model: Optional[FCNRegressor] = field(default=None, init=False, repr=False)
    _y_log: bool = field(default=True, init=False, repr=False)  # всегда используем log1p
    _device: torch.device = field(
        default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        init=False,
        repr=False,
    )

    # -------------------------------------------------------------------------
    # Загрузка данных
    # -------------------------------------------------------------------------
    def load_data(self) -> Tuple[np.ndarray, np.ndarray]:
        """Загружает X и y из .npy, обрабатывает pickle-формат."""
        X = np.load(self.x_path, allow_pickle=True)
        y = np.load(self.y_path, allow_pickle=True).ravel()

        if X.dtype == object:
            X = np.vstack(X).astype(np.float32)
        if y.dtype == object:
            y = np.concatenate(y).astype(np.float32)
        else:
            y = y.astype(np.float32)

        X = X.astype(np.float32)
        return X, y

    # -------------------------------------------------------------------------
    # Предобработка
    # -------------------------------------------------------------------------
    def preprocess(
        self, X: np.ndarray, y: np.ndarray
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Train/test split → StandardScaler на X → log1p на y.

        log1p(y) критически важен для зарплатных данных:
        распределение зарплат логнормальное, MSE в исходном пространстве
        создаёт огромные градиенты от выбросов и мешает обучению.
        """
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state
        )

        self._scaler = StandardScaler()
        X_train = self._scaler.fit_transform(X_train)
        X_test = self._scaler.transform(X_test)

        # log1p transform таргета
        y_train_log = np.log1p(y_train)
        y_test_log = np.log1p(y_test)

        to_t = lambda a: torch.tensor(a, dtype=torch.float32)
        return to_t(X_train), to_t(X_test), to_t(y_train_log), to_t(y_test_log)

    # -------------------------------------------------------------------------
    # DataLoader
    # -------------------------------------------------------------------------
    def make_loader(self, X: torch.Tensor, y: torch.Tensor, shuffle: bool = True) -> DataLoader:
        return DataLoader(TensorDataset(X, y), batch_size=self.batch_size, shuffle=shuffle)

    # -------------------------------------------------------------------------
    # Одна эпоха обучения
    # -------------------------------------------------------------------------
    def _train_epoch(
        self,
        model: FCNRegressor,
        loader: DataLoader,
        optimizer: torch.optim.Optimizer,
        scheduler: Any,
        criterion: nn.Module,
    ) -> float:
        model.train()
        total_loss = 0.0

        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(self._device), y_batch.to(self._device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # градиентный клиппинг
            optimizer.step()
            scheduler.step()  # OneCycleLR обновляется per-batch
            total_loss += loss.item() * len(X_batch)

        return total_loss / len(loader.dataset)

    # -------------------------------------------------------------------------
    # Валидационный loss
    # -------------------------------------------------------------------------
    @torch.no_grad()
    def _val_loss(self, model: FCNRegressor, loader: DataLoader, criterion: nn.Module) -> float:
        model.eval()
        total = 0.0
        for X_b, y_b in loader:
            total += criterion(model(X_b.to(self._device)), y_b.to(self._device)).item() * len(X_b)
        return total / len(loader.dataset)

    # -------------------------------------------------------------------------
    # Метрики (в исходном пространстве через expm1)
    # -------------------------------------------------------------------------
    @torch.no_grad()
    def evaluate(
        self, model: FCNRegressor, X_t: torch.Tensor, y_log_t: torch.Tensor
    ) -> Dict[str, float]:
        """
        Считает метрики в ИСХОДНОМ пространстве (expm1 от предсказаний и таргета).
        R² в исходном пространстве — главный показатель качества.
        """
        model.eval()
        y_pred_log = model(X_t.to(self._device)).cpu().numpy()
        y_true_log = y_log_t.numpy()

        # возвращаем в исходное пространство
        y_pred = np.expm1(y_pred_log)
        y_true = np.expm1(y_true_log)

        # clip отрицательных предсказаний (физически невозможны)
        y_pred = np.clip(y_pred, 0, None)

        mse = mean_squared_error(y_true, y_pred)
        return {
            "mse": float(mse),
            "rmse": float(mse ** 0.5),
            "mae": float(mean_absolute_error(y_true, y_pred)),
            "r2": float(r2_score(y_true, y_pred)),
            "explained_variance": float(explained_variance_score(y_true, y_pred)),
        }

    # -------------------------------------------------------------------------
    # Отчёт
    # -------------------------------------------------------------------------
    @staticmethod
    def _fmt(name: str, value: float, digits: int = 4) -> str:
        return f"{name:25s}: {value:.{digits}f}"

    def report(self, train_m: Dict[str, float], test_m: Dict[str, float]) -> None:
        sep = "=" * 70
        print(f"\n{sep}")
        print("МЕТРИКИ НА TRAIN (исходное пространство)".center(70))
        print("-" * 70)
        for k, v in train_m.items():
            print(self._fmt(k.upper(), v, 4 if k in ("r2", "explained_variance") else 2))
        print(f"\n{'-' * 70}")
        print("МЕТРИКИ НА TEST (исходное пространство)".center(70))
        print("-" * 70)
        for k, v in test_m.items():
            print(self._fmt(k.upper(), v, 4 if k in ("r2", "explained_variance") else 2))
        print(f"{sep}\n")

    # -------------------------------------------------------------------------
    # Предсказание и сохранение
    # -------------------------------------------------------------------------
    @torch.no_grad()
    def predict_and_save(self, X: np.ndarray, output_path: str = "result.npy") -> None:
        if self._best_model is None or self._scaler is None:
            raise AttributeError("Сначала вызови run().")
        X_s = torch.tensor(self._scaler.transform(X), dtype=torch.float32).to(self._device)
        self._best_model.eval()
        y_pred = np.expm1(self._best_model(X_s).cpu().numpy())
        y_pred = np.clip(y_pred, 0, None)
        np.save(output_path, y_pred)
        print(f"[INFO] Предсказания сохранены → {output_path}")

    # -------------------------------------------------------------------------
    # Главный метод
    # -------------------------------------------------------------------------
    def run(self) -> Dict[str, Any]:
        """Полный цикл с early stopping и MLflow-трекингом."""
        torch.manual_seed(self.random_state)
        print(f"[INFO] Устройство: {self._device}")

        # данные
        X_np, y_np = self.load_data()
        print(f"[INFO] X shape: {X_np.shape} | y range: [{y_np.min():.0f}, {y_np.max():.0f}]")

        X_train_t, X_test_t, y_train_t, y_test_t = self.preprocess(X_np, y_np)

        train_loader = self.make_loader(X_train_t, y_train_t, shuffle=True)
        val_loader = self.make_loader(X_test_t, y_test_t, shuffle=False)

        # модель
        input_dim = X_train_t.shape[1]
        model = FCNRegressor(input_dim, self.hidden_dim, self.n_blocks, self.dropout).to(self._device)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"[INFO] Параметров модели: {n_params:,}")

        optimizer = torch.optim.AdamW(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=self.lr,
            steps_per_epoch=len(train_loader),
            epochs=self.max_epochs,
            pct_start=0.1,           # 10% эпох — прогрев
            anneal_strategy="cos",   # косинусный спад
        )
        criterion = nn.HuberLoss(delta=1.0)  # устойчив к выбросам в log-пространстве

        # MLflow
        mlflow.set_tracking_uri(self.mlflow_uri)
        mlflow.set_experiment(self.experiment_name)

        with mlflow.start_run(run_name=self.model_name) as run:

            mlflow.log_params({
                "model_name": self.model_name,
                "hidden_dim": self.hidden_dim,
                "n_blocks": self.n_blocks,
                "dropout": self.dropout,
                "lr": self.lr,
                "weight_decay": self.weight_decay,
                "batch_size": self.batch_size,
                "max_epochs": self.max_epochs,
                "patience": self.patience,
                "loss": "HuberLoss",
                "optimizer": "AdamW",
                "scheduler": "OneCycleLR",
                "log1p_target": True,
                "input_dim": input_dim,
                "n_params": n_params,
            })

            best_val_loss = float("inf")
            best_state = None
            no_improve = 0

            for epoch in range(1, self.max_epochs + 1):
                train_loss = self._train_epoch(model, train_loader, optimizer, scheduler, criterion)
                val_loss = self._val_loss(model, val_loader, criterion)

                # early stopping
                if val_loss < best_val_loss - 1e-6:
                    best_val_loss = val_loss
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    no_improve = 0
                else:
                    no_improve += 1

                if epoch % 20 == 0 or epoch == 1:
                    print(f"Epoch {epoch:4d}/{self.max_epochs}  |  "
                          f"train: {train_loss:.5f}  |  val: {val_loss:.5f}  |  "
                          f"no_improve: {no_improve}/{self.patience}")
                    mlflow.log_metrics(
                        {"train_loss": train_loss, "val_loss": val_loss}, step=epoch
                    )

                if no_improve >= self.patience:
                    print(f"\n[INFO] Early stopping на эпохе {epoch}. Лучший val_loss: {best_val_loss:.5f}")
                    break

            # загружаем лучшие веса
            model.load_state_dict(best_state)
            model.to(self._device)

            # метрики в исходном пространстве
            train_metrics = self.evaluate(model, X_train_t, y_train_t)
            test_metrics = self.evaluate(model, X_test_t, y_test_t)

            for k, v in train_metrics.items():
                mlflow.log_metric(f"{k}_train", v)
            for k, v in test_metrics.items():
                mlflow.log_metric(f"{k}_test", v)

            mlflow.log_metric("r2_score_test", test_metrics["r2"])
            mlflow.pytorch.log_model(model, artifact_path="model")

            run_id = run.info.run_id
            print(f"\n[INFO] MLflow Run ID: {run_id}")

        self._best_model = model
        self.report(train_metrics, test_metrics)

        return {
            "model": model,
            "train_metrics": train_metrics,
            "test_metrics": test_metrics,
            "run_id": run_id,
        }

=============================================================================
# ИСПОЛЬЗОВАНИЕ КЛАССА
=============================================================================

In [22]:
trainer = FCNTrainer(
    x_path="data/X.npy",
    y_path="data/y.npy",
    model_name="ovsyannikov_artem_fcn",
    hidden_dim=256,
    n_blocks=4,
    dropout=0.1,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=512,
    max_epochs=500,
    patience=30,
)

results = trainer.run()

[INFO] Устройство: cuda
[INFO] X shape: (52131, 32) | y range: [1, 194894]
[INFO] Параметров модели: 1,063,425
Epoch    1/500  |  train: 2.16187  |  val: 0.77071  |  no_improve: 0/30
Epoch   20/500  |  train: 0.14499  |  val: 0.17345  |  no_improve: 10/30
Epoch   40/500  |  train: 0.13300  |  val: 0.14842  |  no_improve: 6/30
Epoch   60/500  |  train: 0.11619  |  val: 0.16936  |  no_improve: 26/30

[INFO] Early stopping на эпохе 64. Лучший val_loss: 0.14109


2026/03/08 13:01:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 13:01:10 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/08 13:01:10 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/08 13:01:15 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128).


[INFO] MLflow Run ID: 64773a283faf4320b28cb7a49eb3d3ef
🏃 View run ovsyannikov_artem_fcn at: http://kamnsv.com:55000/#/experiments/129956837527025464/runs/64773a283faf4320b28cb7a49eb3d3ef
🧪 View experiment at: http://kamnsv.com:55000/#/experiments/129956837527025464

               МЕТРИКИ НА TRAIN (исходное пространство)               
----------------------------------------------------------------------
MSE                      : 1240480896.00
RMSE                     : 35220.46
MAE                      : 24814.97
R2                       : 0.5498
EXPLAINED_VARIANCE       : 0.5540

----------------------------------------------------------------------
               МЕТРИКИ НА TEST (исходное пространство)                
----------------------------------------------------------------------
MSE                      : 1388738944.00
RMSE                     : 37265.79
MAE                      : 26242.29
R2                       : 0.5104
EXPLAINED_VARIANCE       : 0.5161



In [23]:
X_full, _ = trainer.load_data()
trainer.predict_and_save(X_full, output_path="result.npy")

print(f"\n✅ Run ID:       {results['run_id']}")
print(f"✅ R² test:      {results['test_metrics']['r2']:.4f}")

[INFO] Предсказания сохранены → result.npy

✅ Run ID:       64773a283faf4320b28cb7a49eb3d3ef
✅ R² test:      0.5104
